# Model Evaluation

## TGB Models

### Load the Dataset and Split

In [68]:
import pandas as pd
df_test = pd.read_parquet('../Data/TGB Preprocessed/df_test.parquet')

print("periodStart: ", min(df_test['periodStart']), ' - ', max( df_test['periodStart']))

print("Shape: ", df_test.shape)

for col in df_test.columns:
    print (f"{col}: ", df_test[col].dtype)

periodStart:  2023-10-26 00:00:00  -  2024-08-29 00:00:00
Shape:  (199848, 483)
id:  category
name:  category
country:  category
latitude:  float32
longitude:  float32
minBorderDistanceKm:  float32
minCapitalDistanceKm:  float32
periodStart:  datetime64[ns]
month_sin:  float32
month_cos:  float32
is_voting:  bool
TargetConflictLocal_t+1:  boolean
TargetConflictRegional_t+1:  boolean
TargetConflictLocal_t+3:  boolean
TargetConflictRegional_t+3:  boolean
TargetConflictLocal_t+6:  boolean
TargetConflictRegional_t+6:  boolean
TargetConflictLocal_t+12:  boolean
TargetConflictRegional_t+12:  boolean
LocalEventCount_std2:  float32
LocalEventCount_std3:  float32
LocalEventCount_std6:  float32
LocalEventCount_std12:  float32
LocalEventCount_std18:  float32
LocalDistinctActorCount_std2:  float32
LocalDistinctActorCount_std3:  float32
LocalDistinctActorCount_std6:  float32
LocalDistinctActorCount_std12:  float32
LocalDistinctActorCount_std18:  float32
LocalDistinctEventTypes_std2:  float32
LocalD

### Load TGB Models from File and Evaluate per Target

In [69]:
#define target columns, we'll iterate over this list later. These need to be exact matches to stop target information leaking into the training (looking at you, AUC = 1) 
target_columns = [
"TargetConflictLocal_t+1",
"TargetConflictRegional_t+1",
"TargetConflictLocal_t+3",
"TargetConflictRegional_t+3",
"TargetConflictLocal_t+6",
"TargetConflictRegional_t+6",
"TargetConflictLocal_t+12",
"TargetConflictRegional_t+12"
]

In [71]:
import joblib
import gc 

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

print("Model Evaluation:")

for target_column in target_columns:

    print(f"\nEvaluating model for: {target_column}")

    #build testing dataset (y; actual target values)
    #test_ids = subsamples_test[target_column]
    #df_test = #df[df["id"].isin(test_ids)].copy()

    y_test = df_test[[target_column]].copy()

    #build the testing dataset (x):
    feature_cols = [
        c for c in df_test.columns
        if c not in target_columns
        and c not in ["id", "placeName", "name", "year", 'latitude', 'longitude', "periodStart", "periodEnd", "period_midpoint"]
    ]

    X_test = df_test[feature_cols].copy()
    
    print("X_test shape:", X_test.shape)
    print("y_test positives:", y_test[target_column].sum())

    #obtain model from dict
    #model = models[target_column]

    #load model from disk (pickle files are neat, this saves tedious mucking around with only saving part of the model):
    model_lgbm = joblib.load(f"../Models/LightGBM/lightgbm_model_{target_column}.pkl")
    model_rf = joblib.load(f"../Models/RF/rf_model_{target_column}.pkl")
    model_catboost = joblib.load(f"../Models/CatBoost/catboost_model_{target_column}.pkl")
    
    #predict
    y_pred_prob_lgbm = model_lgbm.predict_proba(X_test)[:, 1]
    y_pred_lgbm = (y_pred_prob_lgbm >= 0.5).astype(int)

    y_pred_prob_rf = model_rf.predict_proba(X_test)[:, 1]
    y_pred_rf = (y_pred_prob_rf >= 0.5).astype(int)

    y_pred_prob_catboost = model_catboost.predict_proba(X_test)[:, 1]
    y_pred_catboost = (y_pred_prob_catboost >= 0.5).astype(int)

    #Conf matrix
    print("Confusion Matricies")
    print("LightGBM")
    cm_lgbm = confusion_matrix(y_test, y_pred_lgbm) #note this all breaks for regional_12, as there are 0 positive instances. 
    print(cm_lgbm)
    tn_lgbm, fp_lgbm, fn_lgbm, tp_lgbm = cm_lgbm.ravel()


    #specificity = TN / (TN + FP)
    specificity_lgbm = round(tn_lgbm / (tn_lgbm + fp_lgbm) if (tn_lgbm + fp_lgbm) > 0 else 0.0, 2)

    #Classification metrics:
    accuracy_lgbm = round(accuracy_score(y_test, y_pred_lgbm ), 2)
    precision_lgbm  = round(precision_score(y_test, y_pred_lgbm , zero_division=0), 2)
    recall_lgbm  = round(recall_score(y_test, y_pred_lgbm , zero_division=0), 2)
    f1_lgbm  = round(f1_score(y_test, y_pred_lgbm , zero_division=0), 2)
    auc_lgbm  = round(roc_auc_score(y_test, y_pred_prob_lgbm) if y_test[target_column].nunique() > 1 else float("nan"), 2)

    print("\nClassification Report (LightGBM):")
    print(classification_report(y_test, y_pred_lgbm, zero_division=0))

    print("Random Forest")
    cm_rf = confusion_matrix(y_test, y_pred_rf) #note this all breaks for regional_12, as there are 0 positive instances. 
    print(cm_rf)
    tn_rf, fp_rf, fn_rf, tp_rf = cm_rf.ravel()

    #specificity = TN / (TN + FP)
    specificity_rf = tn_rf / (tn_rf + fp_rf) if (tn_rf + fp_rf) > 0 else 0.0

    #Classification metrics:
    accuracy_rf = round(accuracy_score(y_test, y_pred_rf ), 2)
    precision_rf  = round(precision_score(y_test, y_pred_rf , zero_division=0), 2)
    recall_rf  = round(recall_score(y_test, y_pred_rf , zero_division=0), 2)
    f1_rf  = round(f1_score(y_test, y_pred_rf , zero_division=0), 2)
    auc_rf  = round(roc_auc_score(y_test, y_pred_prob_rf) if y_test[target_column].nunique() > 1 else float("nan"), 2)

    print("\nClassification Report (Random Forest):")
    print(classification_report(y_test, y_pred_rf, zero_division=0))
    
    print("CatBoost")
    cm_catboost = confusion_matrix(y_test, y_pred_catboost) #note this all breaks for regional_12, as there are 0 positive instances. 
    print(cm_catboost)
    
    tn_catboost, fp_catboost, fn_catboost, tp_catboost = cm_catboost.ravel()

    #specificity = TN / (TN + FP)
    specificity_catboost = tn_catboost / (tn_catboost + fp_catboost) if (tn_catboost + fp_catboost) > 0 else 0.0

    #Classification metrics:
    accuracy_catboost = round(accuracy_score(y_test, y_pred_catboost ), 2)
    precision_catboost  = round(precision_score(y_test, y_pred_catboost , zero_division=0), 2)
    recall_catboost  = round(recall_score(y_test, y_pred_catboost , zero_division=0), 2)
    f1_catboost  = round(f1_score(y_test, y_pred_catboost , zero_division=0), 2)
    auc_catboost  = round(roc_auc_score(y_test, y_pred_prob_catboost) if y_test[target_column].nunique() > 1 else float("nan"), 2)

    print("\nClassification Report (CatBoost):")
    print(classification_report(y_test, y_pred_catboost, zero_division=0))

    print("\nCombined Classification Metrics")
    #
    #headers = ["Model", "Accuracy", "Specivity", "Precision", "Recall", "F1", "AUC"]
    #rows = [
    #    ["LightGBM", {accuracy_lgbm},{specificity_lgbm}, {precision_lgbm}, {recall_lgbm}, {f1_lgbm}, {auc_lgbm}],
    #    ["Random Forest", {accuracy_rf},{specificity_rf}, {precision_rf}, {recall_rf}, {f1_rf}, {auc_rf}],
    #    ["CatBoost", {accuracy_catboost},{specificity_catboost}, {precision_catboost}, {recall_catboost}, {f1_catboost}, {auc_catboost}]
    #]
    

    results_df = pd.DataFrame([
        {
            "Model": "LightGBM",
            "Accuracy": round(accuracy_lgbm, 2),
            "Specificity": round(specificity_lgbm, 2),
            "Precision": round(precision_lgbm, 2),
            "Recall": round(recall_lgbm, 2),
            "F1": round(f1_lgbm, 2),
            "AUC": round(auc_lgbm, 2)
        },
        {
            "Model": "Random Forest",
            "Accuracy": round(accuracy_rf, 2),
            "Specificity": round(specificity_rf, 2),
            "Precision": round(precision_rf, 2),
            "Recall": round(recall_rf, 2),
            "F1": round(f1_rf, 2),
            "AUC": round(auc_rf, 2)
        },
        {
            "Model": "CatBoost",
            "Accuracy": round(accuracy_catboost, 2),
            "Specificity": round(specificity_catboost, 2),
            "Precision": round(precision_catboost, 2),
            "Recall": round(recall_catboost, 2),
            "F1": round(f1_catboost, 2),
            "AUC": round(auc_catboost, 2)
        }
    ])
    
    results_df.to_csv(f"../Evaluation/model_metrics_{target_column}.csv", index=False)
    print (results_df)

    print("\nFeature Importance")
    
    # Extract top 20 for each model
    fi_lgbm = get_top_features(model_lgbm, feature_cols, "lgbm_importance")
    fi_rf   = get_top_features(model_rf,   feature_cols, "rf_importance")
    fi_cat  = get_top_features(model_catboost, feature_cols, "cat_importance")
     
     # Merge side-by-side on feature name
    fi_merged = (
         fi_lgbm
         .merge(fi_rf, on="feature", how="outer")
         .merge(fi_cat, on="feature", how="outer")
    )

    fi_merged.to_csv(f"../Evaluation/feature_importance_{target_column}.csv", index=False)
    
    print(fi_merged)
   
    gc.collect()

    print(f"End of evaluation for model {target_column}")
    print("------")
    #end of this iteration
    

Model Evaluation:

Evaluating model for: TargetConflictLocal_t+1
X_test shape: (199848, 470)
y_test positives: 18815
Confusion Matricies
LightGBM
[[180352    681]
 [     6  18809]]

Classification Report (LightGBM):
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00    181033
         1.0       0.97      1.00      0.98     18815

    accuracy                           1.00    199848
   macro avg       0.98      1.00      0.99    199848
weighted avg       1.00      1.00      1.00    199848

Random Forest
[[ 17499 163534]
 [   140  18675]]

Classification Report (Random Forest):
              precision    recall  f1-score   support

         0.0       0.99      0.10      0.18    181033
         1.0       0.10      0.99      0.19     18815

    accuracy                           0.18    199848
   macro avg       0.55      0.54      0.18    199848
weighted avg       0.91      0.18      0.18    199848

CatBoost
[[180609    424]
 [     1  18814

NameError: name 'accuracy_lgbm' is not defined

## Chronos 2
For the sake of a simple comparison, we will try to compare the Chonos models on the same testing set as the DT models (2023-10-26 - 2024-08-29)

In [72]:
target_coluns = ['target_conflict_indicator_local', 'target_conflict_indicator_regional']

### Load Dataset

In [73]:
df_full = pd.read_parquet("../Data/EastAfrica Preprocessed/eastafrica.parquet")

In [74]:
##Drop columns named max: suspect leakage.
df_full = df_full.drop(columns=df_full.filter(regex="Max").columns)

#drop the median columns if they still somehow exist:
df_full = df_full.drop(columns=df_full.filter(regex="Median").columns)
df_full = df_full.drop(columns=df_full.filter(regex="median").columns)

In [75]:
#type conversion: id to string. 
df_full["id"] = df_full["id"].astype("string")

In [76]:
#Remove rows with missing data
df_full = df_full.dropna(
    subset=["Armed_forces_personnel__of_total_labor_force_MSMILTOTLTFZS"]
)

In [77]:
#Remove bad Ids with suspect data:

# compute counts
df_counts = (
    df_full.groupby("id", observed=False)
      .size()
      .reset_index(name="count")
)

# get list of bad IDs
bad_ids = df_counts.query("count != 125")["id"].tolist()

# drop rows with those IDs
print("Before deletion:", df_full[df_full["id"].isin(bad_ids)].shape)
df_full.drop(df_full[df_full["id"].isin(bad_ids)].index, inplace=True)
print("After deletion:", df_full[df_full["id"].isin(bad_ids)].shape)


# recompute counts to verify
df_counts = (
    df_full.groupby("id", observed=False)
      .size()
      .reset_index(name="count")
)
df_full.reset_index(drop=True, inplace=True)


bad_ids_after = df_counts.query("count != 125")
print("Bad IDs in df_full:", bad_ids_after["id"].tolist())
print("Counts for bad IDs (df_full):")
print(bad_ids_after)

Before deletion: (2250, 53)
After deletion: (0, 53)
Bad IDs in df_full: []
Counts for bad IDs (df_full):
Empty DataFrame
Columns: [id, count]
Index: []


Loading the full dataset is challenging is there is model training going on at the same time. We can pare down the dataset to 200 random Ids to make sure it works. 


In [78]:
#Local:
df_full['target_conflict_indicator_local'] = (
    (df_full['LocalTotalFatalities'] >= 3) |
    (df_full['LocalMeanSeverity'] >= 3)
).astype(int)

#Regional:
df_full['target_conflict_indicator_regional'] = (
    (df_full['RegionalTotalFatalities'] >= 3) |
    (df_full['RegionalMeanSeverity'] >= 3)
).astype(int)

target_columns = ['target_conflict_indicator_local', 'target_conflict_indicator_regional']

In [79]:
cutoff_lower = df_test["periodStart"].min()
cutoff_upper = df_test["periodStart"].max()

df_context_chronos = df_full[df_full["periodStart"] < cutoff_lower].copy()

df_future_chronos = df_full[
    (df_full["periodStart"] >= cutoff_lower) &
    (df_full["periodStart"] <= cutoff_upper)
].copy()


In [81]:
#Doing the whole dataset is a challenge due to RAM limitations. Randomly sample 200 ids. A different sampling strategy might be needed if they are overwhelmingly peaceful places. 
# Step 1: sample 200 IDs
sample_ids = df_full["id"].drop_duplicates().sample(n=200, random_state=42)

# Step 2: keep all rows for those IDs
df_context_chronos= df_context_chronos[df_context_chronos["id"].isin(sample_ids)].copy()
df_future_chronos = df_future_chronos[df_future_chronos["id"].isin(sample_ids)].copy()


In [82]:
# recompute counts to verify
df_counts = (
    df_future_chronos.groupby("id", observed=False)
      .size()
      .reset_index(name="count")
)
df_future_chronos.reset_index(drop=True, inplace=True)
df_counts

,id,count
0,Central_African_Republic_Abba-Bogani,12
1,Central_African_Republic_Arrondissement_3,12
2,Central_African_Republic_Bamatara,12
3,Central_African_Republic_Bayanga,12
4,Central_African_Republic_Bolom,12
...,...,...
195,Uganda_Metu,12
196,Uganda_Mugoye,12
197,Uganda_Nsiima,12
198,Uganda_Rugyeyo,12


### Load Chronos2 Models and Evaluate

In [84]:
import joblib
print("Model Evaluation:")

for target_column in target_columns:

    print(f"\nEvaluating models for: {target_column}")

    if "local" in target_column.lower():
        local_or_regional = "local"
    if "regional" in target_column.lower():
        local_or_regional = "regional"

    y_test = df_future_chronos[[target_column]].copy()

    #build the testing dataset (x):
    feature_cols = [
        c for c in df_context_chronos.columns
        if c not in target_columns
        and c not in ["id", "placeName", "name", "year", 'latitude', 'longitude', "periodStart", "periodEnd", "period_midpoint"]
    ]

    print("y_test positives:", y_test.sum())


    #load model from disk (pickle files are neat, this saves tedious mucking around with only saving part of the model):
    model_vanilla = joblib.load(f"../Models/Chronos2/chronos2_model_{local_or_regional}_vanilla.pkl")
    model_lora = joblib.load(f"../Models/Chronos2/chronos2_model_{local_or_regional}_lora.pkl")
    #model_finetuned = joblib.load(f"../Models/Chronos2/chronos2_model_{local_or_regional}_full_finetuning.pkl")

    drop_cols = [t for t in target_columns if t != target_column]

    #Vanilla
    pred_df_vanilla= model_vanilla.predict_df(
        df =df_context_chronos.drop(columns=drop_cols).copy(),
        future_df=df_future_chronos.drop(columns=target_columns).copy(), #drop the target cols from the future df; we don't need them. 
        prediction_length=12,  # Number of steps to forecast
        quantile_levels=[0.1, 0.5, 0.9],  # Quantiles for probabilistic forecast; 3x answers per test, model is 10%, 50% and 90% confident the value is below this answer. 
        id_column="id",  # Column identifying different time series
        timestamp_column="periodStart",  # Column with datetime information
        target=target_column,  # Column(s) with time series values to predict
        cross_learning=True,  # Enable cross-learning;
    )
    
    eval_df_vanilla = pred_df_vanilla.merge(
        df_future_chronos[["id", "periodStart", target_column]],
        on=["id", "periodStart"],
        how="inner"
    )
    y_pred_vanilla = eval_df_vanilla[target_column].values          # binary 0/1 target
    y_pred_prob_vanilla = eval_df_vanilla["0.5"].values        # Chronos2 median probability

    #Conf matrix
    print("Confusion Matricies")
    print("Vanilla")
    cm_vanilla = confusion_matrix(y_test, y_pred_vanilla) #note this all breaks for regional_12, as there are 0 positive instances. 
    print(cm_vanilla)
    tn_vanilla, fp_vanilla, fn_vanilla, tp_vanilla = cm_vanilla.ravel()


    #specificity = TN / (TN + FP)
    specificity_vanilla = round(tn_vanilla / (tn_vanilla + fp_vanilla) if (tn_vanilla + fp_vanilla) > 0 else 0.0, 2)

    #Classification metrics:
    accuracy_vanilla = round(accuracy_score(y_test, y_pred_vanilla ), 2)
    precision_vanilla  = round(precision_score(y_test, y_pred_vanilla , zero_division=0), 2)
    recall_vanilla  = round(recall_score(y_test, y_pred_vanilla , zero_division=0), 2)
    f1_vanilla  = round(f1_score(y_test, y_pred_vanilla , zero_division=0), 2)
    auc_vanilla  = round(roc_auc_score(y_test, y_pred_prob_vanilla) if y_test[target_column].nunique() > 1 else float("nan"), 2)

    print("\nClassification Report (Vanilla):")
    print(classification_report(y_test, y_pred_vanilla, zero_division=0))

    print("\nLoRA")

    #LoRA
    pred_df_lora= model_lora.predict_df(
        df =df_context_chronos.drop(columns=drop_cols).copy(),
        future_df=df_future_chronos.drop(columns=target_columns).copy(), #drop the target cols from the future df; we don't need them. 
        prediction_length=12,  # Number of steps to forecast
        quantile_levels=[0.1, 0.5, 0.9],  # Quantiles for probabilistic forecast; 3x answers per test, model is 10%, 50% and 90% confident the value is below this answer. 
        id_column="id",  # Column identifying different time series
        timestamp_column="periodStart",  # Column with datetime information
        target=target_column,  # Column(s) with time series values to predict
        cross_learning=True,  # Enable cross-learning;
    )
    
    eval_df_lora = pred_df_lora.merge(
        df_future_chronos[["id", "periodStart", target_column]],
        on=["id", "periodStart"],
        how="inner"
    )
    y_pred_lora = eval_df_lora[target_column].values          # binary 0/1 target
    y_pred_prob_lora = eval_df_lora["0.5"].values        # Chronos2 median probability
    
    
    cm_lora = confusion_matrix(y_test, y_pred_lora) #note this all breaks for regional_12, as there are 0 positive instances. 
    print(cm_lora)
    tn_lora, fp_lora, fn_lora, tp_lora = cm_lora.ravel()

    #specificity = TN / (TN + FP)
    specificity_lora = tn_lora / (tn_lora + fp_lora) if (tn_lora + fp_lora) > 0 else 0.0

    #Classification metrics:
    accuracy_lora = round(accuracy_score(y_test, y_pred_lora ), 2)
    precision_lora  = round(precision_score(y_test, y_pred_lora , zero_division=0), 2)
    recall_lora  = round(recall_score(y_test, y_pred_lora , zero_division=0), 2)
    f1_lora  = round(f1_score(y_test, y_pred_lora , zero_division=0), 2)
    auc_lora  = round(roc_auc_score(y_test, y_pred_prob_lora) if y_test[target_column].nunique() > 1 else float("nan"), 2)

    print("\nClassification Report (LoRA):")
    print(classification_report(y_test, y_pred_lora, zero_division=0))
    
    print("Fine Tuned")

    pred_df_finetuned= model_finetuned.predict_df(
        df =df_context_chronos.drop(columns=drop_cols).copy(),
        future_df=df_future_chronos.drop(columns=target_columns).copy(), #drop the target cols from the future df; we don't need them. 
        prediction_length=12,  # Number of steps to forecast
        quantile_levels=[0.1, 0.5, 0.9],  # Quantiles for probabilistic forecast; 3x answers per test, model is 10%, 50% and 90% confident the value is below this answer. 
        id_column="id",  # Column identifying different time series
        timestamp_column="periodStart",  # Column with datetime information
        target=target_column,  # Column(s) with time series values to predict
        cross_learning=True,  # Enable cross-learning;
    )
    
    eval_df_finetuned = pred_df_finetuned.merge(
        df_future_chronos[["id", "periodStart", target_column]],
        on=["id", "periodStart"],
        how="inner"
    )
    y_pred_finetuned = eval_df_finetuned[target_column].values          # binary 0/1 target
    y_pred_prob_finetuned = eval_df_finetuned["0.5"].values        # Chronos2 median probability
    
    cm_finetuned = confusion_matrix(y_test, y_pred_finetuned) #note this all breaks for regional_12, as there are 0 positive instances. 
    print(cm_finetuned)
    
    tn_finetuned, fp_finetuned, fn_finetuned, tp_finetuned = cm_finetuned.ravel()

    #specificity = TN / (TN + FP)
    specificity_finetuned = tn_finetuned / (tn_finetuned + fp_finetuned) if (tn_finetuned + fp_finetuned) > 0 else 0.0

    #Classification metrics:
    accuracy_finetuned = round(accuracy_score(y_test, y_pred_finetuned ), 2)
    precision_finetuned  = round(precision_score(y_test, y_pred_finetuned , zero_division=0), 2)
    recall_finetuned  = round(recall_score(y_test, y_pred_finetuned , zero_division=0), 2)
    f1_finetuned  = round(f1_score(y_test, y_pred_finetuned , zero_division=0), 2)
    auc_finetuned  = round(roc_auc_score(y_test, y_pred_prob_finetuned) if y_test[target_column].nunique() > 1 else float("nan"), 2)

    print("\nClassification Report (Fine Tuned):")
    print(classification_report(y_test, y_pred_finetuned, zero_division=0))

    print("\nCombined Classification Metrics")
    
    results_df = pd.DataFrame([
        {
            "Model": "Vanilla",
            "Accuracy": round(accuracy_vanilla, 2),
            "Specificity": round(specificity_vanilla, 2),
            "Precision": round(precision_vanilla, 2),
            "Recall": round(recall_vanilla, 2),
            "F1": round(f1_vanilla, 2),
            "AUC": round(auc_vanilla, 2)
        },
        {
            "Model": "LoRA",
            "Accuracy": round(accuracy_lora, 2),
            "Specificity": round(specificity_lora, 2),
            "Precision": round(precision_lora, 2),
            "Recall": round(recall_lora, 2),
            "F1": round(f1_lora, 2),
            "AUC": round(auc_lora, 2)
        },
        {
            "Model": "Fine Tuned",
            "Accuracy": round(accuracy_finetuned, 2),
            "Specificity": round(specificity_finetuned, 2),
            "Precision": round(precision_finetuned, 2),
            "Recall": round(recall_finetuned, 2),
            "F1": round(f1_finetuned, 2),
            "AUC": round(auc_finetuned, 2)
        }
    ])
    
    results_df.to_csv(f"../Evaluation/Chronos2_model_metrics_{target_column}.csv", index=False)
    print (results_df)

    gc.collect()

    print(f"End of evaluation for model {target_column}")
    print("------")
    #end of this iteration

Model Evaluation:

Evaluating models for: target_conflict_indicator_local
y_test positives: target_conflict_indicator_local    173
dtype: int64
Confusion Matricies
Vanilla
[[2065  162]
 [ 162   11]]

Classification Report (Vanilla):
              precision    recall  f1-score   support

           0       0.93      0.93      0.93      2227
           1       0.06      0.06      0.06       173

    accuracy                           0.86      2400
   macro avg       0.50      0.50      0.50      2400
weighted avg       0.86      0.86      0.86      2400


LoRA
[[2065  162]
 [ 162   11]]

Classification Report (LoRA):
              precision    recall  f1-score   support

           0       0.93      0.93      0.93      2227
           1       0.06      0.06      0.06       173

    accuracy                           0.86      2400
   macro avg       0.50      0.50      0.50      2400
weighted avg       0.86      0.86      0.86      2400

Fine Tuned
[[2065  162]
 [ 162   11]]

Classifica

In [ ]:
pd.Series(y_pred).value_counts()